# Set parameters

In [ ]:
# Build Image
ARTIFACT_REPO= 'ARTIFACT_REPO'
PROJECT_ID= 'PROJECT_ID'
REGION= 'REGION'
DESCRIPTION= 'DESCRIPTION'
IMAGE_NAME= 'IMAGE_NAME'

# Deploy Cloud Run
JOB_NAME= 'JOB_NAME'
SA= 'SA'
LABELS= 'LABELS'
CPU= 'CPU'
MEMORY= 'MEMORY'
TIMEOUT= 'TIMEOUT'
MAX_RETRIES= 'MAX_RETRIES'

# Write based files

In [ ]:
%%writefile Dockerfile
# Use the official lightweight Python image.
FROM python:3.13.1-slim

# Allow statements and log 
ENV PYTHONUNBUFFERED True

# Copy local code to the container image.
ENV APP_HOME /app
WORKDIR $APP_HOME
COPY . ./

# Install production dependencies.
RUN pip install --no-cache-dir -r requirements.txt

# Run
CMD ["sh", "-c", "DATE=${RUN_DT:-$(date +%Y-%m-%d)}; papermill ${NOTEBOOK_BUCKET}/${NOTEBOOK_NAME} ${NOTEBOOK_BUCKET}/${DATE}.ipynb -k python3"]


In [ ]:
%%writefile requirements.txt
papermill
notebook
gcsfs # To Save result into the GCS

# Create artifact registory (1 time)

In [ ]:
!gcloud artifacts repositories create {ARTIFACT_REPO} \
    --repository-format=docker \
    --project={PROJECT_ID} \
    --location={REGION} \
    --description="{DESCRIPTION}" 

# Build Image

In [ ]:
!gcloud builds submit \
    --region {REGION} \
    --tag {REGION}-docker.pkg.dev/{PROJECT_ID}/{ARTIFACT_REPO}/{IMAGE_NAME}

# Deploy

In [ ]:
!echo gcloud run jobs deploy {JOB_NAME} \
    --image={REGION}-docker.pkg.dev/{PROJECT_ID}/{ARTIFACT_REPO}/{IMAGE_NAME} \
    --project={PROJECT_ID} \
    --service-account={SA} \
    --labels={LABELS} \
    --region={REGION} \
    --cpu={CPU} \
    --memory={MEMORY} \
    --task-timeout={TIMEOUT} \
    --max-retries={MAX_RETRIES} 

# Clean Up

In [26]:
import os 
os.remove('Dockerfile')
os.remove('requirements.txt')